# 卵巢癌 Xenium + Flex scRNA-seq 数据预处理

**数据集**
- scRNA-seq：Xenium Flex 平台，`17k_Ovarian_Cancer_scFFPE_count_filtered_feature_bc_matrix.h5`
- 预标注：`FLEX_Ovarian_Barcode_Cluster_Annotation.csv`
- Xenium：`Xenium_V1_Human_Ovarian_Cancer_Addon_FFPE_outs/`

**Cell 执行顺序**
| Cell | 内容 | 输出 |
|---|---|---|
| 1 | 初始化路径 | — |
| 2 | 加载 Flex scRNA + 附加 CSV 注释 | `.rds` 缓存 |
| 3 | UMAP 可视化 | `figures/scrna_umap_ovarian.pdf` |
| 4 | 加载 Xenium 基础对象 | `.rds` 缓存 |
| **4b** | **Seurat 标签转移（TransferData）** | **`seurat_label_transfer_ovarian.csv`** |
| 5 | 完成确认 | — |

⚠️ **Python 训练 notebook 不依赖本 notebook 的 `.rds` 输出**，  
但需要 Cell 4b 生成的 `seurat_label_transfer_ovarian.csv` 用于对比实验。

In [ ]:
# ============================================================
# Cell 1: 初始化
# ============================================================
setwd("/home/ailab/caohao/AdaDiss/")

suppressPackageStartupMessages({
    library(Seurat)
    library(dplyr)
    library(ggplot2)
    library(Matrix)
})

PATHS <- list(
    raw = list(
        flex_h5     = "data/raw/flex/17k_Ovarian_Cancer_scFFPE_count_filtered_feature_bc_matrix.h5",
        flex_annot  = "data/raw/flex/FLEX_Ovarian_Barcode_Cluster_Annotation.csv",
        xenium_dir  = "data/raw/xenium/Xenium_V1_Human_Ovarian_Cancer_Addon_FFPE_outs/"
    ),
    cache = list(
        scrna_annotated = "data/cache/scrna/scrna_annotated_ovarian.rds",
        xenium_base     = "data/cache/xenium/xenium_base_ovarian.rds"
    )
)

for (d in c("data/cache/scrna", "data/cache/xenium")) {
    dir.create(d, recursive = TRUE, showWarnings = FALSE)
}

cat("=== 路径检查 ===\n")
cat("Flex H5   :", file.exists(PATHS$raw$flex_h5),    "\n")
cat("Flex CSV  :", file.exists(PATHS$raw$flex_annot), "\n")
cat("Xenium dir:", dir.exists(PATHS$raw$xenium_dir),  "\n")

In [ ]:
# ============================================================
# Cell 2: 加载 Flex scRNA-seq + 附加预标注
# Flex 平台已提供 cluster annotation CSV，跳过聚类步骤
# ============================================================

if (file.exists(PATHS$cache$scrna_annotated)) {
    cat("[Cache HIT] 加载已注释 scRNA 对象...\n")
    scrna_obj <- readRDS(PATHS$cache$scrna_annotated)
} else {
    cat("[Cache MISS] 加载 Flex H5 文件...\n")

    # ── 加载计数矩阵 ───────────────────────────────────────
    counts <- Read10X_h5(PATHS$raw$flex_h5)
    scrna_obj <- CreateSeuratObject(
        counts   = counts,
        project  = "Ovarian_Flex",
        min.cells = 3, min.features = 200
    )
    cat("  原始细胞数:", ncol(scrna_obj), "\n")
    cat("  基因数    :", nrow(scrna_obj), "\n")

    # ── 附加预标注（直接从 Flex Barcode CSV）─────────────────
    annot <- read.csv(PATHS$raw$flex_annot, stringsAsFactors = FALSE)
    cat("  CSV 列名  :", paste(colnames(annot), collapse = ", "), "\n")
    cat("  CSV 前3行 :\n")
    print(head(annot, 3))

    # 自动识别 barcode 列 和 cell_type 列
    bc_col  <- grep("barcode|cell_id|barcodes", colnames(annot), 
                    ignore.case = TRUE, value = TRUE)[1]
    ct_col  <- grep("cell_type|celltype|annotation|cluster_name|label",
                    colnames(annot), ignore.case = TRUE, value = TRUE)[1]
    cat("  barcode 列:", bc_col, "\n")
    cat("  类型列    :", ct_col, "\n")

    rownames(annot) <- annot[[bc_col]]
    common_bc <- intersect(colnames(scrna_obj), annot[[bc_col]])
    cat("  有标注细胞:", length(common_bc), "/", ncol(scrna_obj), "\n")

    scrna_obj <- scrna_obj[, common_bc]
    scrna_obj$cell_type <- annot[common_bc, ct_col]

    # ── 基础 QC（线粒体过滤）─────────────────────────────────
    scrna_obj[["percent.mt"]] <- PercentageFeatureSet(scrna_obj, pattern = "^MT-")
    scrna_obj <- subset(scrna_obj,
        subset = nFeature_RNA > 200 & nFeature_RNA < 6000 & percent.mt < 25
    )
    cat("  QC 后细胞数:", ncol(scrna_obj), "\n")

    # ── 标准化（供 UMAP 可视化，不影响 GNN 输入）────────────
    scrna_obj <- NormalizeData(scrna_obj, verbose = FALSE)
    scrna_obj <- FindVariableFeatures(scrna_obj, nfeatures = 3000, verbose = FALSE)
    scrna_obj <- ScaleData(scrna_obj, verbose = FALSE)
    scrna_obj <- RunPCA(scrna_obj, npcs = 30, verbose = FALSE)
    scrna_obj <- RunUMAP(scrna_obj, dims = 1:20, verbose = FALSE)

    saveRDS(scrna_obj, PATHS$cache$scrna_annotated)
    cat("[Saved] scRNA 对象 →", PATHS$cache$scrna_annotated, "\n")
}

cat("\n=== 细胞类型分布 ===\n")
print(sort(table(scrna_obj$cell_type), decreasing = TRUE))

In [ ]:
# ============================================================
# Cell 3: UMAP 可视化（检查注释质量）
# ============================================================

p <- DimPlot(scrna_obj, group.by = "cell_type", label = TRUE, repel = TRUE,
             pt.size = 0.3) +
     ggtitle("Flex scRNA-seq — Ovarian Cancer Cell Types") +
     theme(plot.title = element_text(hjust = 0.5))

print(p)
ggsave("figures/scrna_umap_ovarian.pdf", p, width = 10, height = 8)

In [ ]:
# ============================================================
# Cell 4: 加载 Xenium 数据（提取 gene panel 供 Python 使用）
# ============================================================

if (file.exists(PATHS$cache$xenium_base)) {
    cat("[Cache HIT] 加载 Xenium 基础缓存...\n")
    xenium_obj <- readRDS(PATHS$cache$xenium_base)
} else {
    cat("[Cache MISS] 加载 Xenium 数据...\n")
    xenium_obj <- LoadXenium(
        data.dir = PATHS$raw$xenium_dir,
        fov       = "fov"
    )
    xenium_obj <- subset(xenium_obj, nCount_Xenium > 0)
    saveRDS(xenium_obj, PATHS$cache$xenium_base)
    cat("[Saved] Xenium 基础对象 →", PATHS$cache$xenium_base, "\n")
}

cat("  Xenium 细胞数:", ncol(xenium_obj), "\n")
cat("  Xenium 基因数:", nrow(xenium_obj), "\n")

# 共同基因（Python 流水线需要）
common_genes <- intersect(rownames(scrna_obj), rownames(xenium_obj))
cat("  共同基因数  :", length(common_genes), "\n")

In [ ]:
# ============================================================
# Cell 4b: Seurat 标签转移（TransferData）
#
# 流程：
#   1. 在共同基因空间重新做 PCA（不用 harmony，避免维度失真）
#   2. FindTransferAnchors 找锚点细胞对
#   3. TransferData 推断每个 Xenium 细胞的类型
#   4. 结果保存为 CSV，供 Python 训练 notebook 对比用
#
# 输出：data/cache/xenium/seurat_label_transfer_ovarian.csv
#   列：cell_id, x, y, predicted_id, prediction_score
# ============================================================

SEURAT_LT_FILE <- "data/cache/xenium/seurat_label_transfer_ovarian.csv"

if (file.exists(SEURAT_LT_FILE)) {
    cat("[Cache HIT] 加载 Seurat 标签转移缓存...\n")
    lt_df <- read.csv(SEURAT_LT_FILE)
    cat("  共", nrow(lt_df), "个细胞\n")
    cat("  预测类型分布:\n")
    print(sort(table(lt_df$predicted_id), decreasing = TRUE))

} else {
    cat("[Cache MISS] 运行 Seurat 标签转移...\n")

    # ── 1. 共同基因子集 ──────────────────────────────────────
    common_genes <- intersect(rownames(scrna_obj), rownames(xenium_obj))
    cat("  共同基因数:", length(common_genes), "\n")
    if (length(common_genes) < 30) {
        stop("共同基因数过少（", length(common_genes), "），请检查基因命名格式")
    }

    scrna_sub  <- scrna_obj[common_genes, ]
    xenium_sub <- xenium_obj[common_genes, ]

    # ── 2. 在共同基因空间重新做 PCA（核心修复）──────────────
    #    不能用 harmony（全基因空间）投影 Xenium（377 基因）
    #    必须在两者共有基因空间单独做 PCA，维度才一致
    cat("  重新计算共同基因 PCA...\n")
    scrna_sub <- NormalizeData(scrna_sub, verbose = FALSE)
    scrna_sub <- ScaleData(scrna_sub, features = common_genes, verbose = FALSE)
    scrna_sub <- RunPCA(scrna_sub, features = common_genes,
                        npcs = 30, verbose = FALSE)

    xenium_sub <- NormalizeData(xenium_sub, verbose = FALSE)
    xenium_sub <- ScaleData(xenium_sub, features = common_genes, verbose = FALSE)

    # ── 3. 寻找迁移锚点 ─────────────────────────────────────
    cat("  寻找迁移锚点...\n")
    anchors <- FindTransferAnchors(
        reference       = scrna_sub,
        query           = xenium_sub,
        features        = common_genes,
        normalization.method = "LogNormalize",
        reference.reduction  = "pca",   # 共同基因 PCA，非 harmony
        dims            = 1:30,
        verbose         = FALSE
    )
    cat("  锚点数:", nrow(anchors@anchors), "\n")

    # ── 4. k.weight 自适应（锚点少时自动降档，避免报错）───────
    n_anchors <- nrow(anchors@anchors)
    kw        <- min(50, max(5, n_anchors - 1))
    cat("  k.weight =" , kw, "\n")

    # ── 5. 标签转移 ──────────────────────────────────────────
    cat("  TransferData...\n")
    predictions <- TransferData(
        anchorset  = anchors,
        refdata    = scrna_sub$cell_type,
        dims       = 1:30,
        k.weight   = kw,
        verbose    = FALSE
    )
    xenium_sub  <- AddMetaData(xenium_sub, predictions)

    # ── 6. 提取坐标 + 预测结果，保存 CSV ─────────────────────
    coords <- GetTissueCoordinates(xenium_sub)
    lt_df  <- data.frame(
        cell_id          = colnames(xenium_sub),
        x                = coords$x,
        y                = coords$y,
        predicted_id     = xenium_sub$predicted.id,
        prediction_score = xenium_sub$predicted.id.score,
        stringsAsFactors = FALSE
    )
    write.csv(lt_df, SEURAT_LT_FILE, row.names = FALSE)
    cat("[Saved] Seurat 标签转移结果 →", SEURAT_LT_FILE, "\n")
    cat("  共", nrow(lt_df), "个细胞\n")
    cat("  预测类型分布:\n")
    print(sort(table(lt_df$predicted_id), decreasing = TRUE))
    cat("  平均置信度:", round(mean(lt_df$prediction_score, na.rm=TRUE), 3), "\n")
}

# ── 7. 空间可视化（Seurat 细胞级预测）───────────────────────
xenium_obj <- AddMetaData(xenium_obj,
    metadata = setNames(lt_df$predicted_id, lt_df$cell_id),
    col.name = "seurat_predicted_id")

p_lt <- ImageDimPlot(xenium_obj, group.by = "seurat_predicted_id",
                     axes = TRUE, size = 0.5) +
    ggtitle("Seurat Label Transfer — Ovarian Cancer Xenium") +
    theme(plot.title = element_text(hjust = 0.5))
print(p_lt)
ggsave("figures/seurat_label_transfer_spatial.pdf", p_lt, width = 10, height = 9)
cat("空间图已保存 → figures/seurat_label_transfer_spatial.pdf\n")

In [ ]:
# ============================================================
# Cell 5: 完成确认
# ============================================================

cat("=== 卵巢癌 R 预处理完成 ===\n\n")
cat("缓存文件状态：\n")
all_files <- c(
    list(PATHS$cache),
    list(seurat_lt = "data/cache/xenium/seurat_label_transfer_ovarian.csv")
)
for (nm in names(all_files)) {
    f    <- all_files[[nm]]
    ex   <- file.exists(f)
    size <- if (ex) paste0(round(file.size(f)/1e6, 1), " MB") else "不存在"
    cat(sprintf("  %-32s %s  (%s)\n", nm, if (ex) "✅" else "❌", size))
}
cat("\n下一步：运行 train_ovarian.ipynb（Python）\n")
cat("  Cell 9 会加载 seurat_label_transfer_ovarian.csv 与 GNN 结果对比\n")